kernel : Python (Pixi)

#### Main Dataset

In [7]:
%load_ext autoreload
%autoreload 2
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "feng"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")

feng_adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, SERIES + ".h5ad"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "CLDN11",  
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX5", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "SLC2A5", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(feng_adata, SERIES, cell_marker_genes, group = "leiden", project="ANNOTATION")

In [4]:
leiden_to_celltype = {
    "0": "Doublet",
    "1": "Oligodendrocyte",
    "2": "Oligodendrocyte",
    "3": "Doublet",
    "4": "Oligodendrocyte",
    "5": "Oligodendrocyte",
    "6": "Doublet",
    "7": "Oligodendrocyte",
    "8": "Oligodendrocyte",
    "9": "Oligodendrocyte",
    "10": "Oligodendrocyte",
    "11": "ExcitatoryNeuron",
    "12": "Oligodendrocyte",
    "13": "Oligodendrocyte",
    "14": "Oligodendrocyte",
    "15": "Bcell",
    "16": "Endothelial",
    "17": "Microglia",
    "18": "Oligodendrocyte",
    "19": "Oligodendrocyte",
    "20": "Oligodendrocyte",
    "21": "Oligodendrocyte",
    "22": "Oligodendrocyte",
    "23": "Oligodendrocyte",
    "24": "Tcell",
    "25": "OPC",
    "26": "Oligodendrocyte",
    "27": "Oligodendrocyte",
    "28": "InhibitoryNeuron",
    "29": "Oligodendrocyte",
    "30": "Oligodendrocyte",
    "31": "Pericyte+VLMC",
    "32": "Microglia",
    "33": "Oligodendrocyte?",
    "34": "Astrocyte",
    "35": "ExcitatoryNeuron",
    "36": "Oligodendrocyte",
    "37": "Astrocyte",
    "38": "Oligodendrocyte",
    "39": "ExcitatoryNeuron",
    "40": "ExcitatoryNeuron",
    "41": "ExcitatoryNeuron",
    "42": "Microglia",
    "43": "InhibitoryNeuron",
    "44": "Microglia",
    "45": "Microglia",
    "46": "InhibitoryNeuron",
    "47": "Ependymal",
    "48": "InhibitoryNeuron",
    "49": "Oligodendrocyte",
    "50": "InhibitoryNeuron",
    "51": "InhibitoryNeuron",
    "52": "COP",
    "53": "ExcitatoryNeuron",
    "54": "ExcitatoryNeuron",
    "55": "InhibitoryNeuron",
    "56": "ExcitatoryNeuron",
}

leiden_str = feng_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
feng_adata.obs["cell"] = mapped

feng_adata = feng_adata[feng_adata.obs["cell"] != "Doublet"].copy()

In [5]:
feng_adata.write_h5ad(os.path.join(final_h5ad_dir, SERIES + "_final.h5ad"))

In [6]:
pre_h5ad_dir = find_env_dir("PRE_H5AD")

opc = feng_adata[
    (feng_adata.obs["cell"] == "OPC") |
    (feng_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_opc_raw.h5ad"))

oligodendrocyte = feng_adata[feng_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligodendrocyte_raw.h5ad"))

oligo = feng_adata[
    (feng_adata.obs["cell"] == "OPC") |
    (feng_adata.obs["cell"] == "COP") | 
    (feng_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))